In [74]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = '', 
)

cursor = conn.cursor()

In [75]:
cursor.execute('CREATE DATABASE IF NOT EXISTS thrift_db')
cursor.execute('USE thrift_db')

In [76]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS thrift_data (
    item_name VARCHAR(50),
    brand VARCHAR(50),
    category VARCHAR(50),
    item_condition VARCHAR(20),
    source VARCHAR(50),
    cost FLOAT,
    selling_price FLOAT,
    discount FLOAT,
    final_price FLOAT,
    profit FLOAT,
    quantity INT,
    date DATE,
    days_to_sell INT,
    customer_type VARCHAR(20),
    payment_method VARCHAR(20)
)
""")

In [77]:
import pandas as pd

df = pd.read_csv("../data/raw/thrift_data.csv")

In [78]:
cursor.execute('DELETE FROM thrift_data')
conn.commit()

In [79]:
cols = [
    "item_name", "brand", "category", "item_condition", "source",
    "cost", "selling_price", "discount", "final_price", "profit",
    "quantity", "date", "days_to_sell",
    "customer_type", "payment_method"
]

data_to_insert = [tuple(row) for _, row in df[cols].iterrows()]

cursor.executemany("""
INSERT INTO thrift_data (
    item_name, brand, category, item_condition, source,
    cost, selling_price, discount, final_price, profit,
    quantity, date, days_to_sell,
    customer_type, payment_method
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", data_to_insert)

conn.commit()

In [80]:
# Column name debug: SQL has reserved words like 'condition'. It's advisable to not use them in dataset columns or python code
#print(df.columns)

In [81]:
df["date"] = pd.to_datetime(df["date"]).dt.date

In [82]:
# To verify number of rows in dataset
cursor.execute('SELECT COUNT(*) FROM thrift_data')
print(cursor.fetchone())

(200,)


In [83]:
# Most profitable category
cursor.execute(
    """
SELECT category, SUM(profit * quantity) AS total_profit
FROM thrift_data
GROUP BY category
ORDER BY total_profit DESC
"""
)

print(cursor.fetchall())

[('Topwear', 3738.819999694824), ('Outerwear', 2766.4599890708923), ('Bottomwear', 2548.3599997758865), ('Footwear', 2194.1899919509888)]


In [84]:
# Fastest selling items
cursor.execute("""
SELECT item_name, AVG(days_to_sell) AS avg_days
FROM thrift_data
GROUP BY item_name
ORDER BY avg_days ASC
""")

print(cursor.fetchall())

[('Sneakers', Decimal('14.1463')), ('Jeans', Decimal('14.5000')), ('T-Shirt', Decimal('14.6129')), ('Hoodie', Decimal('16.0256')), ('Jacket', Decimal('17.3265'))]


In [85]:
# Best sourcing location
cursor.execute("""
SELECT source, SUM(profit * quantity) AS total_profit
FROM thrift_data
GROUP BY source
ORDER BY total_profit DESC
""")

print(cursor.fetchall())

[('Supplier', 4300.939983844757), ('Donation', 3489.0699887275696), ('Market', 3457.820007920265)]


In [86]:
# Worst performing items
cursor.execute("""
SELECT item_name, AVG(profit) AS avg_profit
FROM thrift_data
GROUP BY item_name
ORDER BY avg_profit ASC
""")

print(cursor.fetchall())

[('Hoodie', 18.109743625689777), ('Jacket', 18.940816222404948), ('T-Shirt', 20.24129020014117), ('Sneakers', 20.278780448727492), ('Jeans', 21.4757500320673)]
